# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook chạy trên Google Colab, tự lưu checkpoint và kết quả vào Google Drive sau từng epoch / thí nghiệm. Nếu runtime bị ngắt, chạy lại từ đầu: dữ liệu đã cache, thí nghiệm hoàn tất được bỏ qua, thí nghiệm dở tiếp tục từ last.pth. Hãy đưa cả bản starter/train.py đã sửa ở local lên Colab/repo trước khi chạy.

**Luật đánh giá:** chọn cấu hình bằng validation; chỉ mở test ở Bước 4, đúng một lần cho mỗi seed. Thiết lập dưới đây là kế hoạch gọn trong 3 giờ, thời lượng thực tế còn phụ thuộc GPU Colab.

Bản đồ: kiểm tra → 5 backbone → T00 + đúng 3 công thức T01–T03 → 4 phương pháp suy luận I01/I02/I04/I07 → final và baseline, 3 seed → sản phẩm.

## 0. Cài đặt môi trường

In [ ]:
# Colab: bật GPU trước khi chạy. Cài dependencies (mất vài phút lần đầu).
!pip -q install timm openpyxl

import os, sys, json, time, hashlib, shutil, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, timm
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR = str(Path.cwd()) if (Path.cwd() / 'eval.py').exists() else '/content/K4-Track4-Day2-Deeplearning-Advance'
if not (Path(REPO_DIR) / 'eval.py').exists():
    raise FileNotFoundError('Hãy clone repo vào /content/K4-Track4-Day2-Deeplearning-Advance trước khi chạy notebook.')
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, str(Path(REPO_DIR) / 'starter'))
from train import Config as TrainConfig
if 'resume_checkpoint' not in TrainConfig.__dataclass_fields__:
    raise RuntimeError('Repo trên Colab chưa có starter/train.py mới từ local; hãy đồng bộ file đó rồi chạy lại.')

DRIVE_ROOT = Path('/content/drive/MyDrive/DeepWeedsLab')
RUNS_DIR, PRED_DIR, CURVES_DIR = DRIVE_ROOT/'runs', DRIVE_ROOT/'predictions', DRIVE_ROOT/'curves'
RESULTS_DIR, DATA_CACHE_DIR = DRIVE_ROOT/'results', DRIVE_ROOT/'data_cache'
EVAL_OUT = str(RESULTS_DIR/'eval_out')
for directory in (RUNS_DIR, PRED_DIR, CURVES_DIR, RESULTS_DIR, DATA_CACHE_DIR): directory.mkdir(parents=True, exist_ok=True)
CODE_BACKUP_DIR = DRIVE_ROOT/'code_snapshot'
CODE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
for source in [Path(REPO_DIR)/'eval.py', *Path(REPO_DIR,'starter').glob('*.py'), Path(REPO_DIR,'starter','lab_day2.ipynb')]:
    if source.exists(): shutil.copy2(source, CODE_BACKUP_DIR/source.name)
TIME_BUDGET_HOURS = 3
SESSION_STARTED = time.monotonic()
DEADLINE = SESSION_STARTED + TIME_BUDGET_HOURS * 3600
STATE_PATH = DRIVE_ROOT/'run_state.json'
print('Python', platform.python_version(), '| torch', torch.__version__, '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU — dừng trước khi train')
if not torch.cuda.is_available(): raise RuntimeError('Bài lab cần GPU; Colab > Runtime > Change runtime type > T4 GPU.')

def atomic_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    os.replace(temp, path)

def seconds_left(): return max(0, DEADLINE-time.monotonic())

def run_logged(exp_id, seed=0, phase='train', **kwargs):
    from train import Config, run
    folder = RUNS_DIR/exp_id/f'seed{seed}'
    result_path, last_path = folder/'result.json', folder/'last.pth'
    state = json.loads(STATE_PATH.read_text(encoding='utf-8')) if STATE_PATH.exists() else {}
    key = f'{exp_id}_seed{seed}'
    cfg_values = dict(exp_id=exp_id, seed=seed, epochs=3, batch_size=32, img_size=224,
                      backbone='convnext_tiny', images_dir=IMAGES_DIR, labels_dir=LABELS_DIR,
                      out_dir=str(RUNS_DIR), pred_dir=str(PRED_DIR), curves_dir=str(CURVES_DIR),
                      num_workers=2, amp=True)
    cfg_values.update(kwargs)
    cfg_values.setdefault('resume_checkpoint', None)
    requested_config = dict(cfg_values)
    requested_config['resume_checkpoint'] = None
    requested_config.setdefault('save_test_predictions', False)
    saved_config = state.get(key, {}).get('config')
    if saved_config:
        saved_config = dict(saved_config)
        saved_config['resume_checkpoint'] = None
        saved_config['save_test_predictions'] = requested_config['save_test_predictions']
        if saved_config != requested_config and state.get(key, {}).get('status') == 'completed':
            raise ValueError(f'{key} đã hoàn tất với cấu hình khác; dùng exp_id mới để tránh ghi đè.')
    test_path = PRED_DIR/f'{exp_id}_seed{seed}_test.csv'
    if (state.get(key, {}).get('status') == 'completed' and result_path.exists()
            and (not requested_config['save_test_predictions'] or test_path.exists())):
        return json.loads(result_path.read_text(encoding='utf-8'))
    if seconds_left() < 60: raise TimeoutError('Còn dưới 60 giây trong ngân sách 3 giờ; dừng an toàn giữa các thí nghiệm.')
    if last_path.exists(): cfg_values['resume_checkpoint'] = str(last_path)
    atomic_json(STATE_PATH, {**state, key:{'status':'running','phase':phase,'updated':time.time(), 'config':cfg_values}})
    result = run(Config(**cfg_values))
    state = json.loads(STATE_PATH.read_text(encoding='utf-8'))
    state[key] = {**state.get(key, {}),'status':'completed','phase':phase,'updated':time.time(),'result':str(result_path)}
    atomic_json(STATE_PATH, state)
    return result

def stage_done(name, details=None):
    state = json.loads(STATE_PATH.read_text(encoding='utf-8')) if STATE_PATH.exists() else {}
    state[f'stage_{name}'] = {'status':'completed','updated':time.time(),'details':details or {}}
    atomic_json(STATE_PATH, state)

print('Drive autosave:', DRIVE_ROOT, '| remaining min:', round(seconds_left()/60, 1))

### Dữ liệu và cache bền vững
ZIP được kiểm MD5 trước khi cache trên Drive. File ảnh/nhãn giải nén tại /content để tăng tốc đọc; runtime sau sẽ khôi phục từ cache Drive.

In [ ]:
# Ưu tiên bản cache Drive; nếu chưa có thì tải một lần rồi cache.
LOCAL_DATA = Path('/content/deepweeds_data')
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
os.makedirs(LOCAL_DATA/'labels', exist_ok=True)
drive_zip = DATA_CACHE_DIR/'images.zip'
local_zip = LOCAL_DATA/'images.zip'
if drive_zip.exists(): shutil.copy2(drive_zip, local_zip)
elif not local_zip.exists():
    !wget -q -O /content/deepweeds_data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"
h = hashlib.md5()
with local_zip.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1 << 20), b''): h.update(chunk)
assert h.hexdigest() == 'b7b30f96d466fba86016aa5a26606e0f', f'MD5 sai: {h.hexdigest()}'
if not drive_zip.exists(): shutil.copy2(local_zip, drive_zip)
!unzip -q -n /content/deepweeds_data/images.zip -d /content/deepweeds_data/

BASE = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
for name in ['labels','train_subset0','val_subset0','test_subset0']:
    cached = DATA_CACHE_DIR/f'{name}.csv'
    local = LOCAL_DATA/'labels'/f'{name}.csv'
    if cached.exists(): shutil.copy2(cached, local)
    elif not local.exists():
        !wget -q -O /content/deepweeds_data/labels/{name}.csv {BASE}/{name}.csv
        shutil.copy2(local, cached)
image_candidates = [p for p in LOCAL_DATA.rglob('*.jpg')]
if not image_candidates: image_candidates = [p for p in LOCAL_DATA.rglob('*.JPG')]
if not image_candidates: raise FileNotFoundError('Không tìm thấy ảnh .jpg sau khi giải nén.')
IMAGES_DIR = str(image_candidates[0].parent)
LABELS_DIR = str(LOCAL_DATA/'labels')
print('Ảnh:', IMAGES_DIR, '| số file mẫu:', len(image_candidates), '| nhãn:', LABELS_DIR)
stage_done('data', {'images_dir':IMAGES_DIR,'labels_dir':LABELS_DIR})

In [ ]:
# Thư mục đầu ra đều nằm trên Drive; checkpoint last.pth + best.pth được cập nhật mỗi epoch.
print('runs:', RUNS_DIR, '| predictions:', PRED_DIR, '| curves:', CURVES_DIR)

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
from dataset import load_split, check_split
from train import set_seed
import matplotlib.pyplot as plt

train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_report = check_split(train_df, val_df, test_df, IMAGES_DIR)
print('Split:', {k:len(v) for k,v in [('train',train_df),('val',val_df),('test',test_df)]})
print('Leakage / file checks:', split_report)
ax = train_df['Label'].value_counts().sort_index().plot.bar(figsize=(8,3), title='Train labels')
fig = ax.get_figure(); fig.tight_layout(); fig.savefig(DRIVE_ROOT/'train_class_counts.png', dpi=140); plt.show()
stage_done('eda', {'split_sizes':[len(train_df),len(val_df),len(test_df)]})

In [ ]:
# Pipeline sanity check: determinism, 9-class head shape and initial CE near ln(9).
from model import build_model
set_seed(17)
probe = build_model('resnet18', pretrained=False, num_classes=9).cuda().eval()
x = torch.randn(2,3,224,224,device='cuda')
with torch.no_grad(): logits = probe(x)
initial_ce = torch.nn.functional.cross_entropy(logits, torch.tensor([0,1],device='cuda')).item()
assert logits.shape == (2,9) and np.isfinite(initial_ce)
print('head shape:', tuple(logits.shape), '| random-init CE (expected order ln(9)=2.197):', round(initial_ce,3))
del probe, x, logits
torch.cuda.empty_cache()
stage_done('pipeline_smoke', {'status':'passed','note':'forward/head/loss; full overfit probe omitted to preserve 3h budget'})

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
from train import Config
from model import SUGGESTED_BACKBONES

# Sàng lọc 5 backbone, một seed, cùng T00. 2 epoch/model để kịp giới hạn runtime.
BACKBONES = ['resnet50','resnext50_32x4d','convnext_tiny','deit_small_patch16_224','efficientnet_b0']
backbone_rows = []
for index, backbone in enumerate(BACKBONES, 1):
    result = run_logged(f'B{index:02d}', phase='backbone', backbone=backbone, epochs=2)
    backbone_rows.append(result)
backbone_table = pd.DataFrame(backbone_rows).sort_values('macro_f1_val', ascending=False)
display(backbone_table[['exp_id','backbone','params_m','gmac','macro_f1_val','top1_val','train_seconds_per_epoch']])
atomic_json(RESULTS_DIR/'backbones.json', backbone_rows)
stage_done('backbone', {'experiments':[f'B{i:02d}' for i in range(1,6)]})
BEST_BACKBONE = str(backbone_table.iloc[0]['backbone'])
print('Chọn theo val:', BEST_BACKBONE)

## Bước 2 — Công thức huấn luyện: đúng 3 biến thể so với T00
Giữ backbone và các siêu tham số nền cố định; chỉ đổi lần lượt augmentation ColorJitter, label smoothing, balanced sampler. Cùng ngân sách epoch và seed để so sánh công bằng.

In [ ]:
# T00 là baseline; T01-T03 là đúng 3 trục/biến thể đã chốt.
# Nếu runtime trước đây đã ghi kết quả ở Drive, run_logged sẽ bỏ qua thí nghiệm hoàn tất.
RECIPE_CONFIGS = {
    'T00': {},
    'T01': {'aug':'color'},                    # ColorJitter
    'T02': {'loss':'ls','label_smoothing':0.1}, # label smoothing
    'T03': {'sampler':'balanced'},              # balanced sampler
}
training_rows = []
for exp_id, changes in RECIPE_CONFIGS.items():
    training_rows.append(run_logged(exp_id, phase='training_recipe', backbone=BEST_BACKBONE,
                                    seed=0, epochs=3, **changes))
training_table = pd.DataFrame(training_rows).sort_values('macro_f1_val', ascending=False)
display(training_table[['exp_id','backbone','macro_f1_val','top1_val','ece_val','train_seconds_per_epoch']])
atomic_json(RESULTS_DIR/'training.json', training_rows)
stage_done('training_recipes', {'experiments':list(RECIPE_CONFIGS)})
BEST_RECIPE = str(training_table.iloc[0]['exp_id'])

## Bước 3 — Suy luận: I01, I02, I04, I07; chọn trên VAL
Chạy inference-only trên checkpoint recipe tốt nhất. I01 flip TTA, I02 5-crop, I04 FixRes 256, I07 temperature scaling; latency có warmup 10, ít nhất 50 lần, batch 1 và batch 32. Test chưa được đọc.

In [ ]:
from inference import run_inference_only

best_record = next(row for row in training_rows if row['exp_id'] == BEST_RECIPE)
CHECKPOINT_PATH = None
CHECKPOINT_PATH = best_record['checkpoint']
inference_result = run_inference_only(CHECKPOINT_PATH, BEST_BACKBONE, IMAGES_DIR, LABELS_DIR,
                                     RESULTS_DIR/'inference', batch_size=32, num_workers=2)
inference_rows = inference_result['rows']
required_methods = {'I01','I02','I04','I07'}
present = {row['exp_id'] for row in inference_rows}
assert required_methods <= present, f'Methods còn thiếu: {required_methods-present}'
atomic_json(RESULTS_DIR/'inference_summary.json', inference_result)
display(pd.DataFrame(inference_rows)[lambda frame: frame['exp_id'].isin(['I00','I01','I02','I04','I07'])])
stage_done('inference', {'required_methods':sorted(required_methods),'selected':inference_result['inference_exp_id']})
# Latency evaluates batch 1 and batch 32; benchmark uses 10 warmups and 50 measured iterations.

## Bước 4 — Chung kết: 3 seed; test đúng một lần mỗi seed
Chạy baseline T00 và cấu hình cuối F01 với seed 0, 1, 2. Mỗi run tự lưu checkpoint theo epoch; cờ test chỉ bật tại đây. Nếu đã có dự đoán test thì không ghi đè/chạy lại.

In [ ]:
# Final và baseline đều dùng cùng backbone/seed; F01 dùng công thức đã chọn trên val.
# Giới hạn ngắn 3 epoch/run để hoàn thành đủ 3 seed trong quota; có thể tăng nếu GPU còn thời gian.
final_recipe = RECIPE_CONFIGS[BEST_RECIPE]
final_rows = []
for seed in (0,1,2):
    for exp_id, recipe in [('T00',{}), ('F01',final_recipe)]:
        test_file = PRED_DIR/f'{exp_id}_seed{seed}_test.csv'
        result_file = RUNS_DIR/exp_id/f'seed{seed}'/'result.json'
        if test_file.exists():
            if result_file.exists(): final_rows.append(json.loads(result_file.read_text(encoding='utf-8')))
            else: print('Đã có test output; không chạy lại:', test_file)
            continue
        final_rows.append(run_logged(exp_id, seed=seed, phase='final_test', backbone=BEST_BACKBONE,
                                     epochs=3, save_test_predictions=True, **recipe))
atomic_json(RESULTS_DIR/'final.json', final_rows)
stage_done('final', {'seeds':[0,1,2],'test_runs_per_seed':1})

In [ ]:
# Điểm test chỉ tổng hợp sau khi cell chung kết đã tạo đủ file; output đặt trên Drive.
!python {REPO_DIR}/eval.py score --pred "{PRED_DIR}/F01_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out {EVAL_OUT}
!python {REPO_DIR}/eval.py score --pred "{PRED_DIR}/T00_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out {EVAL_OUT}

In [ ]:
!python {REPO_DIR}/eval.py grade --final "{PRED_DIR}/F01_seed*_test.csv" --baseline "{PRED_DIR}/T00_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out {EVAL_OUT}

## Bước 5 — Sản phẩm được ghi trực tiếp lên Drive
`results.xlsx`, `report.md`, curves, checkpoints, stage state và predictions đều nằm trong `MyDrive/DeepWeedsLab`.

In [ ]:
# Bảng tổng hợp audit; các kết quả gốc JSON/CSV và checkpoint cũng đã nằm trên Drive.
tables = {
    'Backbones': pd.DataFrame(backbone_rows),
    'Training': pd.DataFrame(training_rows),
    'Inference': pd.DataFrame(inference_rows),
    'Final': pd.DataFrame(final_rows),
}
with pd.ExcelWriter(DRIVE_ROOT/'results.xlsx', engine='openpyxl') as writer:
    for sheet, frame in tables.items(): frame.to_excel(writer, sheet_name=sheet, index=False)
report = [
    '# DeepWeeds Lab — kết quả tự động',
    f'- Backbone chọn trên val: {BEST_BACKBONE}',
    f'- Công thức cuối: {BEST_RECIPE}',
    f'- Phương pháp suy luận chọn trên val: {inference_result["inference_exp_id"]}',
    '- Test: 3 seed, một lần mỗi seed cho F01 và T00.',
    '- Thời gian chạy được cấu hình tối đa 3 giờ; phụ thuộc GPU Colab.',
]
(DRIVE_ROOT/'report.md').write_text('\n'.join(report)+'\n', encoding='utf-8')
stage_done('deliverables', {'xlsx':str(DRIVE_ROOT/'results.xlsx'),'report':str(DRIVE_ROOT/'report.md')})
print('Đã lưu hoàn tất:', DRIVE_ROOT)